# Ableitungsspektren

**Autor:** jd · **Datum:** 2026-10-29 · **Issue:** –

**Hypothese:** Die erste Ableitung entlang der Wellenlänge behält die Form (z. B. den Anstieg der Red Edge), entfernt aber einen gleichmäßigen Helligkeitsversatz wie einen hellen Bodenhintergrund. Anders als SNV (−0.024) verwirft sie nicht die ganze Helligkeit; zusätzlich wird sie neben den Spektren und mit Albedo geprüft. Ableitungen verstärken Rauschen, deshalb auch mit Savitzky-Golay-Glättung (Fenster 7) davor.

**Ablauf:** Jede Variante bekommt dasselbe `C` × `gamma`-Raster wie die Referenz (`combined_svm_aez_month`). Entschieden wird nur per CV; validiert wird nur, wenn der Gewinn `MIN_CV_GAIN` (0.005) erreicht.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from awp2.config import CROP_COL, MIN_CV_GAIN, STAGE_COL
from awp2.data import load_split
from awp2.evaluation import compare_predictions, recall_by_class
from awp2.experiment import RunConfig, TuneConfig, out_of_fold_predictions, run, tune
from awp2.models.combined import make_combined_svm
from awp2.plots import plot_cv_scores, plot_metric_comparison, plot_recall_comparison
from awp2.preprocessing import PreprocessingConfig

# A · Läufe

In [ ]:
# reference: combined_svm_aez_month (d93cea89); every variant gets the same grid
REFERENCE = PreprocessingConfig(use_spectral_standard_scale=True, use_aez_month=True)
GRID = {"estimator__C": [100.0, 300.0, 1000.0], "estimator__gamma": [0.001, 0.003, 0.01]}
y_train = load_split().y_train
SLUG = "combined_svm_derivative"
TOPIC = "Ableitungsspektren"
SMOOTH = {"use_savgol": True, "savgol_window_length": 7, "savgol_polyorder": 2}
REPLACE = {"use_spectra": False, "use_spectral_derivative": True, "use_mean_albedo": True}
VARIANTS = {
    "reference": ("Referenz", REFERENCE),
    "append": ("+ Ableitung", REFERENCE.model_copy(update={"use_spectral_derivative": True})),
    "append_smooth": (
        "+ Ableitung, geglättet",
        REFERENCE.model_copy(update={"use_spectral_derivative": True, **SMOOTH}),
    ),
    "replace": ("Ableitung + Albedo statt Spektren", REFERENCE.model_copy(update=REPLACE)),
    "replace_smooth": (
        "Ableitung + Albedo statt Spektren, geglättet",
        REFERENCE.model_copy(update={**REPLACE, **SMOOTH}),
    ),
}

In [ ]:
searches = {
    label: tune(
        make_combined_svm(),
        TuneConfig(
            name=f"{SLUG}_{key}_search",
            approach="combined",
            description=f"{TOPIC}: {label}; RBF-SVM, gleiches Raster wie die Referenz.",
            param_grid=GRID,
            preprocessing=preprocessing,
        ),
    )
    for key, (label, preprocessing) in VARIANTS.items()
}
pd.DataFrame(
    {label: {"CV": result.best_score, **result.best_params} for label, result in searches.items()}
).T

# B · Erkenntnisse

## 1. CV-Score je Variante

In [ ]:
reference_cv = searches["Referenz"].best_score
ax = plot_cv_scores(
    {label: result.candidates for label, result in searches.items()}, reference=reference_cv
)
ax.set_title(f"{TOPIC}: bester CV-Score je Variante (gestrichelt: Referenz)")

## 2. Gepaart auf dem Trainingsteil (out-of-fold)

Gleiche Zeilen, beste Variante gegen die Referenz: Wie viele Zeilen trifft nur eine?

In [ ]:
variants = [label for label in searches if label != "Referenz"]
best_label = max(variants, key=lambda label: searches[label].best_score)
best = searches[best_label]
oof = {
    label: out_of_fold_predictions(searches[label].best_model, searches[label].config)
    for label in ("Referenz", best_label)
}
print(f"Beste Variante: {best_label}")
compare_predictions(y_train, oof["Referenz"], oof[best_label])

## 3. Zielfehler: Weizen → Baumwolle (helle Felder) und früher Mais ↔ Soja

In [ ]:
early = y_train[STAGE_COL].isin(["Emerge_VEarly", "Early_Mid"])


def target_errors(pred: pd.DataFrame) -> pd.Series:
    return pd.Series(
        {
            "Weizen → Baumwolle": (
                (y_train[CROP_COL] == "winter_wheat") & (pred[CROP_COL] == "cotton")
            ).sum(),
            "früher Mais ↔ Soja": (
                early
                & y_train[CROP_COL].isin(["corn", "soybean"])
                & pred[CROP_COL].isin(["corn", "soybean"])
                & (y_train[CROP_COL] != pred[CROP_COL])
            ).sum(),
        }
    )


pd.DataFrame({label: target_errors(pred) for label, pred in oof.items()})

## 4. Recall je Klasse: wo ändert sich etwas?

In [ ]:
plot_recall_comparison(
    {label: recall_by_class(y_train, pred) for label, pred in oof.items()}, top=10
)

## 5. Entscheidung

In [ ]:
gain = best.best_score - reference_cv
print(f"CV-Gewinn {gain:+.4f} (Schwelle {MIN_CV_GAIN})")
if gain >= MIN_CV_GAIN:
    validated = run(
        best.best_model,
        RunConfig(
            name=f"{SLUG}",
            approach="combined",
            description=f"{TOPIC}: {best_label}, einmal auf der Validierung.",
            preprocessing=best.config.preprocessing,
            tuning_run=best.run_id,
        ),
    )
    reference = run(
        searches["Referenz"].best_model,
        RunConfig(name="reference", approach="combined", preprocessing=REFERENCE, track=False),
    )
    display(plot_metric_comparison({"Referenz": reference.metrics, best_label: validated.metrics}))
else:
    print("Kein Gewinn über der Rauschschwelle – keine Validierung, Referenz bleibt.")

# Fazit

| Variante | CV |
| --- | --- |
| Referenz | **0.864** |
| + Ableitung, geglättet | 0.804 |
| + Ableitung | 0.791 |
| Ableitung + Albedo statt Spektren, geglättet | 0.779 |
| Ableitung + Albedo statt Spektren | 0.765 |

- **Insgesamt klar schlechter** (−0.06 und mehr): 131 zusätzliche, verrauschte Spalten verzerren die
  Abstände des RBF-Kernels; Glättung mildert das nur.
- **Aber der Helligkeitsbefund bestätigt sich:** Weizen → Baumwolle fällt von 60 auf 42 Fehler.
  Die hellen Weizenfelder sind tatsächlich ein Niveau-Problem; der Gewinn dort wird von
  Verlusten anderswo (u. a. früher Mais ↔ Soja 64 → 73) mehr als aufgezehrt.
- Keine Validierung; die Referenz bleibt.